# 02 — Train/Test Split & Naive Baseline

**Goal:**
1. Clean (drop duplicates) and create a **stratified** train/test split that every later notebook reuses.
2. Show *why* naive modelling fails on imbalanced data, using two baselines:
   - `DummyClassifier` — always predicts "legit"
   - Plain `LogisticRegression` — no imbalance handling

**Rule for the whole project:** the test set is locked away until `05_threshold`. All model comparison happens with **cross-validation on the training set**.

In [1]:
import sys
sys.path.append("..")   # so `src` is importable from notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from src.config import PROCESSED_DIR, REPORTS_DIR, TARGET, RANDOM_STATE, TEST_SIZE
from src.utils import load_raw, build_pipeline, cv_evaluate, get_cv

## 1. Clean: drop duplicates

In [2]:
df = load_raw()
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df):,} duplicates -> {len(df):,} rows, {df[TARGET].sum()} frauds")

Dropped 1,081 duplicates -> 283,726 rows, 473 frauds


## 2. Stratified train/test split

`stratify=y` guarantees both sets keep the ~0.17% fraud ratio. Without it, a random split could leave the test set with noticeably fewer (or more) frauds, making results unreliable.

In [3]:
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

split_info = pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "frauds": [y_train.sum(), y_test.sum()],
    "fraud_%": [y_train.mean() * 100, y_test.mean() * 100],
}, index=["train", "test"])
split_info

,rows,frauds,fraud_%
train,226980,378,0.1665
test,56746,95,0.1674


In [4]:
# Save splits so every later notebook uses exactly the same data
pd.concat([X_train, y_train], axis=1).to_pickle(PROCESSED_DIR / "train.pkl")
pd.concat([X_test, y_test], axis=1).to_pickle(PROCESSED_DIR / "test.pkl")
print("Saved train.pkl and test.pkl to", PROCESSED_DIR)

Saved train.pkl and test.pkl to /tmp/claude-0/-home-claude/25bce366-f326-5915-ae9f-7aeaf528573c/scratchpad/real/data/processed


## 3. Baseline 1 — "always legit" (DummyClassifier)

This is the trap: sky-high accuracy, zero usefulness.

In [5]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_pred = dummy.predict(X_train)
print(f"Accuracy: {accuracy_score(y_train, dummy_pred):.4%}")
print(f"Frauds caught: {((dummy_pred == 1) & (y_train == 1)).sum()} of {y_train.sum()}")

Accuracy: 99.8335%
Frauds caught: 0 of 378


## 4. Baseline 2 — plain Logistic Regression (5-fold stratified CV)

The pipeline is `RobustScaler(Time, Amount) → LogisticRegression`. Because the scaler is *inside* the pipeline, `cross_validate` refits it on each training fold only — the validation fold never leaks into the scaling statistics.

In [6]:
baseline_pipe = build_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
baseline = cv_evaluate(baseline_pipe, X_train, y_train, name="LogReg (no imbalance handling)")
dummy_res = cv_evaluate(DummyClassifier(strategy="prior"), X_train, y_train, name="Dummy (always legit)")

results = pd.DataFrame([dummy_res, baseline]).set_index("name")
results[["pr_auc", "roc_auc", "precision", "recall", "f1"]]

,pr_auc,roc_auc,precision,recall,f1
name,,,,,
Dummy (always legit),0.0017,0.5000,0.0000,0.0000,0.0000
LogReg (no imbalance handling),0.7527,0.9764,0.8676,0.6136,0.7168


### How to read this (what you should typically see)

- **Dummy:** PR-AUC ≈ 0.0017 — that is exactly the fraud rate, the score of random guessing. ROC-AUC = 0.5 (random). Recall = 0.
- **Plain LogReg:** ROC-AUC looks excellent (often ~0.97), but **recall is noticeably lower** — at the default 0.5 threshold the model misses a good share of frauds, because the loss is dominated by the 99.8% legit class.
- **ROC-AUC vs PR-AUC:** ROC-AUC uses the False Positive Rate = FP / (FP + TN). With ~227k negatives, even thousands of false alarms barely move FPR, so ROC-AUC looks great. PR-AUC uses *precision* = TP / (TP + FP), which feels every false alarm. That's why PR-AUC is our primary metric.

This baseline is the bar every technique in `03` must beat.

In [7]:
results.to_csv(REPORTS_DIR / "02_baseline_results.csv")
print("Saved baseline results.")

Saved baseline results.
